# 1. Title and context — Demo pipeline (archival)

**Question:** Does **`evaluate_batch`** in **`canonical_full_mode`** exercise gate failure, approval with abstention fields, and abstention-driven rejection on **fixed** triple of cases?

**Output:** `outputs/figures/demo_pipeline_summary.txt` only.


# 2. Why this matters

Full mode adds **SCM abstention** and **fallback adequacy** on top of gates and compensatory scoring. The summary file makes branch behaviour **human-readable** for audit while the digest fingerprints the entire batch object.


# 3. Deterministic setup

Dependencies include **ipywidgets**. **Canonical write** uses the fixed `ARCHIVAL_CASES` list below—never the dropdown selection.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

OUT_FIG = ROOT / "outputs" / "figures"
OUT_FIG.mkdir(parents=True, exist_ok=True)


# 4. Inputs — archival batch + per-case exploration index


In [2]:
ARCHIVAL_CASES = [
    {
        "case_id": "demo_replay_pass",
        "features": {
            "intrinsic_safety": 0.62,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.58,
            "traceability_integrity": 0.57,
        },
    },
    {
        "case_id": "demo_gate_fail",
        "features": {
            "intrinsic_safety": 0.35,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.58,
            "traceability_integrity": 0.57,
        },
    },
    {
        "case_id": "demo_fullmode_abstention",
        "features": {
            "intrinsic_safety": 0.62,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.42,
            "traceability_integrity": 0.50,
            "fallback_safety_delta": 0.15,
        },
    },
]


# 5. Modular evaluation


In [3]:
def evaluate_full_single(case: dict) -> dict:
    # Single-case canonical_full_mode evaluation (public API).
    return eng.evaluate_case(case, profile_name="moderate", mode=eng.MODE_CANONICAL_FULL)


# 6. Interactive — inspect one archival case at a time (display only)


In [4]:
idx_out = widgets.Output(layout={"border": "1px solid #ccc", "padding": "6px"})
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(ARCHIVAL_CASES) - 1,
    step=1,
    description="Case index:",
    continuous_update=False,
)


def _render_idx(i: int) -> None:
    with idx_out:
        clear_output(wait=True)
        case = ARCHIVAL_CASES[i]
        r = evaluate_full_single(case)
        print(f"case_id={case['case_id']}")
        print(f"  outcome: {r['governance_outcome']}  approved={r['approved']}")
        if "abstention_rate" in r:
            print(
                f"  abstention_rate={r['abstention_rate']} "
                f"triggered={r['abstention_triggered']} "
                f"fallback_adequate={r['fallback_adequate']}"
            )


def _on_s(change) -> None:
    if change.get("name") == "value":
        _render_idx(int(change["new"]))


slider.observe(_on_s, names="value")
_render_idx(int(slider.value))
display(slider, idx_out)


IntSlider(value=0, continuous_update=False, description='Case index:', max=2)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# 7. Output generation — demo summary file (strict)


In [5]:
CASES = ARCHIVAL_CASES

batch = eng.evaluate_batch(CASES, profile_names=["moderate"], mode=eng.MODE_CANONICAL_FULL)
approved = sum(
    1 for cid in batch if batch[cid]["profiles"]["moderate"]["approved"]
)
rejected = len(batch) - approved
digest = eng.hash_output(batch)

lines = [
    "Shared-core demo pipeline (03_demo_pipeline)",
    "============================================",
    "",
    "Purpose: This notebook runs a small, fixed batch of cases in canonical_full_mode,",
    "so both non-compensatory gates and SCM-derived abstention / fallback logic are",
    "exercised alongside the compensatory layer.",
    "",
    "How it works: Each case is a dict consumed by the engine's evaluate_batch helper.",
    "The moderate threshold profile from the engine's canonical table is used.",
    "",
    "Why this matters: The batch shows contrasting outcomes (approval with defaults,",
    "hard gate failure, and abstention with inadequate fallback) using the same public",
    "entry points the shared core documents.",
    "",
    "Output (this notebook only):",
    "- outputs/figures/demo_pipeline_summary.txt",
    "",
    f"Cases in batch:           {len(CASES)}",
    f"Approved (moderate):      {approved}",
    f"Rejected (moderate):      {rejected}",
    f"Batch digest (SHA-256):   {digest}",
    "",
    "Per-case governance outcomes (moderate profile):",
]
for c in CASES:
    r = batch[c["case_id"]]["profiles"]["moderate"]
    extra = ""
    if "abstention_rate" in r:
        extra = (
            f" abstention_rate={r['abstention_rate']} "
            f"abstention_triggered={r['abstention_triggered']} "
            f"fallback_adequate={r['fallback_adequate']}"
        )
    lines.append(
        f"- {c['case_id']}: {r['governance_outcome']} (approved={r['approved']}){extra}"
    )

out_path = OUT_FIG / "demo_pipeline_summary.txt"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Demo pipeline summary written.")


Demo pipeline summary written.


# 8. Interpretation

Counts and per-case lines reflect **moderate** thresholds and full-mode abstention logic. The batch digest changes if any case dict or engine behaviour drifts—hence hash locking.

# 9. Reproducibility

`ARCHIVAL_CASES` alone drives the summary file. Slider exploration is **non-destructive**. Run **`python reproduce_all.py`** for manifest validation against `config/expected_outputs.json`.
